<a href="https://colab.research.google.com/github/raqueldatainfo/APILanchonete/blob/main/C%C3%B3pia_de_C%C3%B3pia_de_Fine_tune_Phi_3_5_Unsloth_Provas_OAB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Instala dependências

Aqui instalamos o `unsloth` e outras bibliotecas necessárias.

In [ ]:
%%capture
# Installs Unsloth, Xformers (Flash Attention) and all other packages!
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

# We have to check which Torch version for Xformers (2.3 -> 0.0.27)
from torch import __version__; from packaging.version import Version as V
xformers = "xformers==0.0.27" if V(__version__) < V("2.4.0") else "xformers"
!pip install --no-deps {xformers} trl peft accelerate bitsandbytes triton

# Carrega dados

Aqui fazemos a carga das provas da OAB resolvidas por LLM que vamos utilizar no treinamento e teste.

*Questões anuladas pela oab foram removidas*

In [ ]:
# this will connect this session to your google drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
df = pd.read_parquet('/content/drive/MyDrive/UN/curso-fine-tuning-unsloth-phi-3.5/data/oab-dataset.parquet')

print(df.NAME_TEST.value_counts())
display(df)

NAME_TEST
XXXI EXAME DE ORDEM UNIFICADO      80
XXVII EXAME DE ORDEM UNIFICADO     79
XXVIII EXAME DE ORDEM UNIFICADO    79
XXXIII EXAME DE ORDEM UNIFICADO    79
XXXIV EXAME DE ORDEM UNIFICADO     79
XXIX EXAME DE ORDEM UNIFICADO      78
37º EXAME DE ORDEM UNIFICADO       78
36º EXAME DE ORDEM UNIFICADO       78
35º EXAME DE ORDEM UNIFICADO       78
XXX EXAME DE ORDEM UNIFICADO       77
39º EXAME DE ORDEM UNIFICADO       77
38º EXAME DE ORDEM UNIFICADO       77
40º EXAME DE ORDEM UNIFICADO       77
XXXII EXAME DE ORDEM UNIFICADO     75
Name: count, dtype: int64


,QUESTION,OPTION_A,OPTION_B,OPTION_C,OPTION_D,OPTION_E,QUESTION_NUMBER,PAGE,ORIGINAL_TOPIC,TOPIC_INFERENCE,...,MODEL_RESPONSE_DATE,ANSWER_GABARITO,MODEL_RESPONSE_ANSWER,MODEL_PROMPT_TYPE,PROMPT,PROMPT_TOKENS,COMPLETION_TOKENS,TOTAL_TOKENS,MODEL_RESPONSE_ANSWER_PARSED,CORRECT
0,"Guilherme é bacharel em Direito, não inscrito ...",Guilherme pode impetrar habeas corpus em favor...,Guilherme pode impetrar habeas corpus em favor...,Guilherme pode impetrar habeas corpus em favor...,Guilherme pode impetrar mandado de segurança e...,None,1,3,Estatuto da Advocacia e da OAB e Código de Ética,Extração manual arquivos .txt /data/gabaritos-...,...,2024-08-29,A,"Ao analisar a questão apresentada, é necessári...",EXPLAINATION_PROMPT,Instrução:\nVocê é um candidato na prova de um...,479,563,1042,B,0
1,"Ricardo Silva, Carlos Santos e Raul Azevedo sã...","É permitida, desde que expressamente autorizad...","É vedada, pois da razão social não pode consta...","É permitida, desde que prevista tal possibilid...","É permitida, independentemente da previsão no ...",None,2,3,Estatuto da Advocacia e da OAB e Código de Ética,Extração manual arquivos .txt /data/gabaritos-...,...,2024-08-29,C,A questão trata da possibilidade de manutenção...,EXPLAINATION_PROMPT,Instrução:\nVocê é um candidato na prova de um...,379,530,909,C,1
2,"O advogado Nelson celebrou, com determinado cl...",O contrato celebrado viola o disposto no Códig...,O contrato celebrado viola o disposto no Códig...,O Código de Ética e Disciplina da OAB autoriza...,O Código de Ética e Disciplina da OAB autoriza...,None,3,3,Estatuto da Advocacia e da OAB e Código de Ética,Extração manual arquivos .txt /data/gabaritos-...,...,2024-08-29,C,"Ao analisar as alternativas, é necessário cons...",EXPLAINATION_PROMPT,Instrução:\nVocê é um candidato na prova de um...,574,517,1091,C,1
3,"A advogada Mariana, gestante, ao ingressar em ...",Mariana tem o direito de não ser submetida a a...,Mariana tem o direito de não ser submetida a a...,"Mariana deverá, por medida de segurança, passa...","Mariana tem o direito, independentemente do te...",None,4,3,Estatuto da Advocacia e da OAB e Código de Ética,Extração manual arquivos .txt /data/gabaritos-...,...,2024-08-29,D,Ao analisar as alternativas à luz do Estatuto ...,EXPLAINATION_PROMPT,Instrução:\nVocê é um candidato na prova de um...,375,499,874,B,0
4,"O advogado Mário dos Santos, presidente do Con...",Compete ao Conselho Seccional Y da OAB promove...,Compete ao Conselho Federal da OAB promover o ...,Compete ao Conselho Seccional Y da OAB promove...,Compete ao Conselho Federal da OAB promover o ...,None,5,4,Estatuto da Advocacia e da OAB e Código de Ética,Extração manual arquivos .txt /data/gabaritos-...,...,2024-08-29,D,"Ao analisar as alternativas fornecidas, é nece...",EXPLAINATION_PROMPT,Instrução:\nVocê é um candidato na prova de um...,371,394,765,A,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1086,"Manoel, empresário do segmento de alimentação,...","Devido à atividade beneficente de Manoel, ele ...",Na condição de empresário administrador de sua...,"Manoel, na condição de administrador de sua so...","Manoel, caso encerre suas atividades profissio...",None,70,19,Direito Previdenciário,Extração manual arquivos .txt /data/gabaritos-...,...,2024-08-29,B,"Ao analisar as alternativas apresentadas, é ne...",EXPLAINATION_PROMPT,Instrução:\nVocê é um candidato na prova de um...,421,562,983,A,0
1087,"Marina, empregada doméstica, é casada com Pedr...",Pedro não possui a carência mínima para recebe...,Marina e Pedro não fazem jus ao salário-famíli...,Marina e Pedro têm direito ao benefício do sal...,"Pedro, na qualidade de trabalhador avulso, não...",None,69,19,Direito Previdenciário,Extração manual arquivos .txt /data/gabaritos-...,...,2024-08-29,C,A questão trata do benefício previdenciário co...,EXPLAINATION_PROMPT,Instrução:\nVocê é um candidato na prova de um...,351,485,836,C,1
1088,Henrique e Amanda foram casados por 30

## Train / Test Split

Aqui dividimos os dados de treino e teste, nesta aula, vamos utilizar as provas anteriores a como base de treinamento e medir a melhora na performance das provas 39 e 40.


### Teste
O conjunto de teste vão ser as provas 39 e 40.

* 39º EXAME DE ORDEM UNIFICADO
* 40º EXAME DE ORDEM UNIFICADO

In [ ]:
test_exams = [
    '39º EXAME DE ORDEM UNIFICADO',
    '40º EXAME DE ORDEM UNIFICADO',
]

test = df[df['NAME_TEST'].isin(test_exams)].reset_index(drop=True)
test.shape

(154, 45)

### Treino
O conjunto de treino serão o restante das provas e suas respectivas respostas, vamos utilizar somente as questões que o LLM resolveu corretamente.

In [ ]:
train = df[~df['NAME_TEST'].isin(test_exams)]
train = train[train['CORRECT']==1].reset_index(drop=True)
train.shape

(719, 45)

# Carrega modelo

Aqui carregamos o modelo para a memória, este é o modelo base.

In [ ]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 2048 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Phi-3.5-mini-instruct",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    # token = "hf_...", # use one if using gated models like meta-llama/Llama-2-7b-hf
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
==((====))==  Unsloth 2024.8: Fast Llama patching. Transformers = 4.44.2.
   \\   /|    GPU: Tesla T4. Max memory: 14.748 GB. Platform = Linux.
O^O/ \_/ \    Pytorch: 2.4.0+cu121. CUDA = 7.5. CUDA Toolkit = 12.1.
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.27.post2. FA2 = False]
 "-____-"     Free Apache license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.26G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/140 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.37k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

## Cria prompt de inferência

Aqui temos um prompt simples para realização de cada questão.

In [ ]:
inference_prompt = """Instrução:
Você é um candidato na prova de um concurso público de direito, aja como um especialista em direito realizando a prova.
Você receberá uma questão de múltipla escolha e terá que assinalar a alternativa correta, inicie a resposta com a letra da resposta correta.
IMPORTANTE: Sua resposta deve ser somente uma letra no seguinte formato: Resposta: A
IMPORTANTE: Não adicione nenhum caractere especial na resposta, como #, *, espaço em branco, \\n, etc.
"""

def prepare_zero_shot_prompt(row):

    question = f'{inference_prompt}'

    # add apply to multiple
    if not pd.isna(row['APPLY_TO_MULTIPLE_TEXT']):
        question += f"Cabeçalho:\n"
        question += f"{row['APPLY_TO_MULTIPLE_TEXT']}\n"

    # prepare question
    question += f"Questão {row['QUESTION_NUMBER']}:\n{row['QUESTION']}\n"

    question += 'Alternativas:\n'

    # add options
    for option in ['OPTION_A', 'OPTION_B', 'OPTION_C', 'OPTION_D', 'OPTION_E']:
        if not pd.isna(row[option]):
            question += f'{option[-1]}. {row[option]}\n'

    question += 'Resposta: '

    return question

test['PROMPT'] = test.apply(prepare_zero_shot_prompt, axis=1)
print(test.iloc[0].PROMPT)

Instrução:
Você é um candidato na prova de um concurso público de direito, aja como um especialista em direito realizando a prova.
Você receberá uma questão de múltipla escolha e terá que assinalar a alternativa correta, inicie a resposta com a letra da resposta correta.
IMPORTANTE: Sua resposta deve ser somente uma letra no seguinte formato: Resposta: A
IMPORTANTE: Não adicione nenhum caractere especial na resposta, como #, *, espaço em branco, \n, etc.
Questão 1:
Mariana deseja ingressar no quadro da Sociedade de Advogados XYZ, na qualidade de associada, sem vínculo de emprego. Ao pesquisar a legislação que rege a parceria em questão, Mariana descobriu que constitui cláusula essencial do contrato de associação
Alternativas:
A. a qualificação das partes, com referência expressa à inscrição no Conselho Seccional da OAB competente.
B. a identificação da parte que terá a responsabilidade exclusiva pelos riscos e pelas receitas decorrentes da prestação do serviço.
C. a forma de repartição

# Resolve TEST com o modelo base

Aqui resolvemos o conjunto de teste com o modelo base. Estamos interessados no valor de `acurácia`

In [ ]:
# Prepare the model for inference
model = FastLanguageModel.for_inference(model)
model

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(32064, 3072)
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear4bit(in_features=3072, out_features=3072, bias=False)
          (k_proj): Linear4bit(in_features=3072, out_features=3072, bias=False)
          (v_proj): Linear4bit(in_features=3072, out_features=3072, bias=False)
          (o_proj): Linear4bit(in_features=3072, out_features=3072, bias=False)
          (rotary_emb): LongRopeRotaryEmbedding()
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear4bit(in_features=3072, out_features=8192, bias=False)
          (up_proj): Linear4bit(in_features=3072, out_features=8192, bias=False)
          (down_proj): Linear4bit(in_features=8192, out_features=3072, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm((3072,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((3072,), eps=1e-0

In [ ]:
from tqdm import tqdm
tqdm.pandas()

def get_response(question):
    # Tokenize the input question
    input_ids = tokenizer.encode(question, return_tensors="pt")

    # Generate a response
    with torch.no_grad():
        output_ids = model.generate(input_ids, max_new_tokens=3)

    # Decode the generated tokens back to text
    response = tokenizer.decode(output_ids[0], skip_special_tokens=True)

    # Remove the input question from the response
    response = response[len(question):].strip()

    return response

test['BASE_PHI_RESPONSE'] = test['PROMPT'].progress_apply(get_response)
test

100%|██████████| 154/154 [01:39<00:00,  1.56it/s]


,QUESTION,OPTION_A,OPTION_B,OPTION_C,OPTION_D,OPTION_E,QUESTION_NUMBER,PAGE,ORIGINAL_TOPIC,TOPIC_INFERENCE,...,ANSWER_GABARITO,MODEL_RESPONSE_ANSWER,MODEL_PROMPT_TYPE,PROMPT,PROMPT_TOKENS,COMPLETION_TOKENS,TOTAL_TOKENS,MODEL_RESPONSE_ANSWER_PARSED,CORRECT,BASE_PHI_RESPONSE
0,Mariana deseja ingressar no quadro da Sociedad...,"a qualificação das partes, com referência expr...",a identificação da parte que terá a responsabi...,a forma de repartição da responsabilidade pelo...,"a estabilidade da parceria, materializada na a...",None,1,3,Estatuto da Advocacia e da OAB e Código de Ética,Extração manual arquivos .txt /data/gabaritos-...,...,A,"Ao analisar as alternativas apresentadas, é im...",EXPLAINATION_PROMPT,Instrução:\nVocê é um candidato na prova de um...,328,395,723,A,1,C.
1,"Alice Santos, advogada, está sendo investigada...",Poderá efetuar colaboração premiada contra Leo...,Poderá efetuar colaboração premiada contra Rob...,Caso efetue colaboração premiada contra Robson...,Caso efetue colaboração premiada contra Leonar...,None,2,3,Estatuto da Advocacia e da OAB e Código de Ética,Extração manual arquivos .txt /data/gabaritos-...,...,D,"Ao analisar as alternativas apresentadas, é ne...",EXPLAINATION_PROMPT,Instrução:\nVocê é um candidato na prova de um...,374,595,969,B,0,B.
2,Durante audiência de instrução e julgamento da...,O relator deverá solicitar informações da auto...,Não há previsão legal ou regulamentar de prazo...,O desagravo será concedido em sessão realizada...,"A sessão de desagravo deverá ser realizada, pr...",None,3,3,Estatuto da Advocacia e da OAB e Código de Ética,Extração manual arquivos .txt /data/gabaritos-...,...,D,"Ao analisar as alternativas apresentadas, é im...",EXPLAINATION_PROMPT,Instrução:\nVocê é um candidato na prova de um...,383,542,925,D,1,B.
3,"Pedro, cidadão brasileiro, graduou-se em Direi...",Revalidação do título de graduação em Direito.,Aprovação em Exame de Ordem.,Ter sido admitido em estágio profissional de a...,Prestação de compromisso perante o conselho.,None,4,3,Estatuto da Advocacia e da OAB e Código de Ética,Extração manual arquivos .txt /data/gabaritos-...,...,C,"Ao analisar as alternativas apresentadas, é ne...",EXPLAINATION_PROMPT,Instrução:\nVocê é um candidato na prova de um...,270,489,759,A,0,A
4,O advogado Edson foi contratado para prestar a...,Edson pode prestar a assessoria de modo verbal...,Edson deve prestar a assessoria de modo escrit...,Edson pode prestar a assessoria de modo verbal...,Edson deve prestar a assessoria de modo escrit...,None,5,3,Estatuto da Advocacia e da OAB e Código de Ética,Extração manual arquivos .txt /data/gabaritos-...,...,A,"Ao analisar as alternativas apresentadas, é im...",EXPLAINATION_PROMPT,Instrução:\nVocê é um candidato na prova de um...,317,476,793,C,0,C.
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
149,"A Escolinha do Gol, entidade privada sem fins ...","Por não se tratar de uma entidade pública, e s...",Não pode o TCU fiscalizar e controlar tais rep...,É devida a prestação de contas de qualquer pes...,Apenas deverão prestar contas dos recursos púb...,None,24,8,Direito Financeiro,Extração manual arquivos .txt /data/gabaritos-...,...,C,"Ao analisar as alternativas apresentadas, é ne...",EXPLAINATION_PROMPT,Instrução:\nVocê é um candidato na prova de um...,561,508,1069,C,1,C.
150,"Marina, empregada doméstica, é casada com Pedr...",Pedro não possui a carência mínima para recebe...,Marina e Pedro não fazem jus ao salário-famíli...,Marina e Pedro têm direito ao benefício do sal...,"Pedro, na qualidade de trabalhador avulso, não...",None,69,19,Direito Previdenciário,Extração manual arquivos .txt /data/gabaritos-...,...,C,A questão trata do benefício previdenciário co...,EXPLAINATION_PROMPT,Instrução:\nVocê é um candidato na prova de um...,351,485,836,C,1,C.
151,Henrique e Amanda foram casados por 30 anos. E...,"Do requerimento, já que foi requerido 60 dias ...","Do óbito, já que foi requerido em até 90 dia

## Parsing da resposta
Aqui conferimos a resposta do modelo versus o gabarito.

In [ ]:
import re
from sklearn.metrics import accuracy_score

def get_answer_from_response(s):
    # gets the first letter of the response
    try:
        return re.search(r'\b[A-E]\b', s).group()
    except:
        return 'ERROR'

test['BASE_PHI_RESPONSE_PARSED'] = test['BASE_PHI_RESPONSE'].apply(get_answer_from_response)
test['CORRECT'] = test['BASE_PHI_RESPONSE_PARSED'] == test['ANSWER']

print(test.BASE_PHI_RESPONSE_PARSED.value_counts())
print('accuracy:', accuracy_score(test['ANSWER'], test['BASE_PHI_RESPONSE_PARSED']))

BASE_PHI_RESPONSE_PARSED
D        50
C        43
B        38
A        22
ERROR     1
Name: count, dtype: int64
accuracy: 0.43506493506493504


In [ ]:
test[test['BASE_PHI_RESPONSE_PARSED']=='ERROR'].BASE_PHI_RESPONSE

,BASE_PHI_RESPONSE
116,#


In [ ]:
test.groupby('NAME_TEST')['CORRECT'].mean()

,CORRECT
NAME_TEST,
39º EXAME DE ORDEM UNIFICADO,0.467532
40º EXAME DE ORDEM UNIFICADO,0.402597


### Resultado Modelo Base

A acurácia do modelo base no conjunto de teste foi de **43.5%**

# Cria o modelo com Fine Tuning

Aqui vamos realizar o fine tuning utilizando o conjunto de **Treino**

## Formatação dos dados para fine tuning

Vamos formatar os dados de acordo com o padrão do `Phi-3`, que é da seguinte forma:
```
<|user|>
Olá, esta é a mensagem do usuário<|end|>
<|assistant|>
Excelente, esta é a mensagem do modelo!<|end|>
```

## Formata mensagem do usuário

Aqui vamos alterar a instrução inicial para levar em consideração a explicação da resposta, isto foi feito como uma tentativa de adicionar mais conhecimento específico para o modelo, não somente a letra da resposta correta.

In [ ]:
training_prompt = """Instrução:
Você é um candidato na prova de um concurso público de direito, aja como um especialista em direito realizando a prova.
Você receberá uma questão de múltipla escolha e terá que assinalar a alternativa correta, inicie a resposta com a letra da resposta correta.
IMPORTANTE: Sua resposta deve ser somente uma letra no seguinte formato: Resposta: A
IMPORTANTE: Não adicione nenhum caractere especial na resposta, como #, *, espaço em branco, \\n, etc.
"""

def prepare_trainig_prompt(row):

    question = f'{inference_prompt}'

    # add apply to multiple
    if not pd.isna(row['APPLY_TO_MULTIPLE_TEXT']):
        question += f"Cabeçalho:\n"
        question += f"{row['APPLY_TO_MULTIPLE_TEXT']}\n"

    # prepare question
    question += f"Questão {row['QUESTION_NUMBER']}:\n{row['QUESTION']}\n"

    question += 'Alternativas:\n'

    # add options
    for option in ['OPTION_A', 'OPTION_B', 'OPTION_C', 'OPTION_D', 'OPTION_E']:
        if not pd.isna(row[option]):
            question += f'{option[-1]}. {row[option]}\n'

    question += "Resposta: "

    return question

train['PROMPT'] = train.apply(prepare_trainig_prompt, axis=1)
print(train.iloc[0].PROMPT)

Instrução:
Você é um candidato na prova de um concurso público de direito, aja como um especialista em direito realizando a prova.
Você receberá uma questão de múltipla escolha e terá que assinalar a alternativa correta, inicie a resposta com a letra da resposta correta.
IMPORTANTE: Sua resposta deve ser somente uma letra no seguinte formato: Resposta: A
IMPORTANTE: Não adicione nenhum caractere especial na resposta, como #, *, espaço em branco, \n, etc.
Questão 2:
Ricardo Silva, Carlos Santos e Raul Azevedo são advogados e constituem a sociedade Silva, Santos e Azevedo Sociedade de Advogados, para exercício conjunto da profissão. A sociedade consolida-se como referência de atuação em determinado ramo do Direito. Anos depois, Carlos Santos falece e seus ex- sócios pretendem manter seu sobrenome na sociedade. Sobre a manutenção do sobrenome de Carlos Santos na sociedade, de acordo com o Estatuto e com o Regulamento Geral da OAB, assinale a afirmativa correta.
Alternativas:
A. É permitid

## Formata mensagem do modelo

Aqui formatamos a resposta que o modelo deve responder.

*Esta etapa foi feita anteriormente e as explicações do modelo foram salvos na coluna MODEL_RESPONSE_ANSWER*

In [ ]:
print(train.iloc[0].ANSWER)

C


# Formata input de treinamento do modelo

Aqui juntamos a mensagem do usuário e a mensagem do modelo.

In [ ]:
# Define the function to format the conversation
def format_conversation(row):
    prompt = row['PROMPT']
    answer = row['ANSWER']
    conversation = {
        'conversations': [
            {'from': 'human', 'value': prompt},
            {'from': 'gpt', 'value': answer}
        ],
        'text': f"<|user|>\n{prompt}<|end|>\n<|assistant|>\n{answer}<|end|>\n"
    }
    return conversation

train['TRAIN_DATA'] = train.apply(format_conversation, axis=1)
train.iloc[0]['TRAIN_DATA']

{'conversations': [{'from': 'human',
   'value': 'Instrução:\nVocê é um candidato na prova de um concurso público de direito, aja como um especialista em direito realizando a prova.\nVocê receberá uma questão de múltipla escolha e terá que assinalar a alternativa correta, inicie a resposta com a letra da resposta correta.\nIMPORTANTE: Sua resposta deve ser somente uma letra no seguinte formato: Resposta: A\nIMPORTANTE: Não adicione nenhum caractere especial na resposta, como #, *, espaço em branco, \\n, etc.\nQuestão 2:\nRicardo Silva, Carlos Santos e Raul Azevedo são advogados e constituem a sociedade Silva, Santos e Azevedo Sociedade de Advogados, para exercício conjunto da profissão. A sociedade consolida-se como referência de atuação em determinado ramo do Direito. Anos depois, Carlos Santos falece e seus ex- sócios pretendem manter seu sobrenome na sociedade. Sobre a manutenção do sobrenome de Carlos Santos na sociedade, de acordo com o Estatuto e com o Regulamento Geral da OAB, a

## Configura pipeline

Aqui temos as configurações de fine tuning, não é necessário alterar nada

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

Unsloth 2024.8 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


## Realiza Fine Tuning

Aqui realizamos o fine tuning, é a etapa lenta.

In [ ]:
import pandas as pd
from datasets import Dataset
from trl import SFTConfig, SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported
import json


# Extract the 'text' field from the 'FORMATTED' column
train['text'] = train['TRAIN_DATA'].apply(lambda x: x['text'])

# Convert the DataFrame to a datasets.Dataset
dataset = Dataset.from_pandas(train[['text']])

# Define the SFT configuration
sft_config = SFTConfig(
    dataset_text_field="text",
    max_seq_length=2048,
    output_dir="/tmp",
)

# Initialize the SFTTrainer
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=2048,
    dataset_num_proc=2,
    packing=False,  # Can make training 5x faster for short sequences.
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        #max_steps=1,
        num_train_epochs=4,
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="outputs",
    ),
)

# Train the model
trainer.train()

Map (num_proc=2):   0%|          | 0/719 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs = 1
   \\   /|    Num examples = 719 | Num Epochs = 4
O^O/ \_/ \    Batch size per device = 2 | Gradient Accumulation steps = 4
\        /    Total batch size = 8 | Total steps = 360
 "-____-"     Number of trainable parameters = 29,884,416


Step,Training Loss
1,1.852900
2,1.954700
3,1.800800
4,1.883400
5,1.873400
6,1.727000
7,1.672500
8,1.572200
9,1.563600
10,1.520500


TrainOutput(global_step=360, training_loss=0.922067306107945, metrics={'train_runtime': 3177.5613, 'train_samples_per_second': 0.905, 'train_steps_per_second': 0.113, 'total_flos': 3.969621416613888e+16, 'train_loss': 0.922067306107945, 'epoch': 4.0})

In [ ]:
#@title Show stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory         /max_memory*100, 3)
lora_percentage = round(used_memory_for_lora/max_memory*100, 3)
#print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
#print(f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training.")
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

GPU = Tesla T4. Max memory = 14.748 GB.
3.035 GB of memory reserved.
Peak reserved memory = 3.035 GB.
Peak reserved memory for training = 0.0 GB.
Peak reserved memory % of max memory = 20.579 %.
Peak reserved memory for training % of max memory = 0.0 %.


Resolve TEST com o modelo fine tuned

Aqui resolvemos as provas novamente, porém com o modelo com fine tuning

In [ ]:
# Prepare the model for inference
model = FastLanguageModel.for_inference(model)
model

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(32064, 3072)
        (layers): ModuleList(
          (0-31): 32 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=3072, out_features=3072, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=3072, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=3072, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.Linear4bit(
       

In [ ]:
def get_response(question):
    # Tokenize the input question
    input_ids = tokenizer.encode(question, return_tensors="pt")

    # Generate a response
    with torch.no_grad():
        output_ids = model.generate(input_ids, max_new_tokens=3)

    # Decode the generated tokens back to text
    response = tokenizer.decode(output_ids[0], skip_special_tokens=True)

    # Remove the input question from the response
    response = response[len(question):].strip()

    return response

test['FINE_TUNED_PHI_RESPONSE'] = test['PROMPT'].progress_apply(get_response)
test

100%|██████████| 154/154 [01:19<00:00,  1.93it/s]


,QUESTION,OPTION_A,OPTION_B,OPTION_C,OPTION_D,OPTION_E,QUESTION_NUMBER,PAGE,ORIGINAL_TOPIC,TOPIC_INFERENCE,...,MODEL_PROMPT_TYPE,PROMPT,PROMPT_TOKENS,COMPLETION_TOKENS,TOTAL_TOKENS,MODEL_RESPONSE_ANSWER_PARSED,CORRECT,BASE_PHI_RESPONSE,BASE_PHI_RESPONSE_PARSED,FINE_TUNED_PHI_RESPONSE
0,Mariana deseja ingressar no quadro da Sociedad...,"a qualificação das partes, com referência expr...",a identificação da parte que terá a responsabi...,a forma de repartição da responsabilidade pelo...,"a estabilidade da parceria, materializada na a...",None,1,3,Estatuto da Advocacia e da OAB e Código de Ética,Extração manual arquivos .txt /data/gabaritos-...,...,EXPLAINATION_PROMPT,Instrução:\nVocê é um candidato na prova de um...,328,395,723,A,False,C.,C,C
1,"Alice Santos, advogada, está sendo investigada...",Poderá efetuar colaboração premiada contra Leo...,Poderá efetuar colaboração premiada contra Rob...,Caso efetue colaboração premiada contra Robson...,Caso efetue colaboração premiada contra Leonar...,None,2,3,Estatuto da Advocacia e da OAB e Código de Ética,Extração manual arquivos .txt /data/gabaritos-...,...,EXPLAINATION_PROMPT,Instrução:\nVocê é um candidato na prova de um...,374,595,969,B,False,B.,B,A
2,Durante audiência de instrução e julgamento da...,O relator deverá solicitar informações da auto...,Não há previsão legal ou regulamentar de prazo...,O desagravo será concedido em sessão realizada...,"A sessão de desagravo deverá ser realizada, pr...",None,3,3,Estatuto da Advocacia e da OAB e Código de Ética,Extração manual arquivos .txt /data/gabaritos-...,...,EXPLAINATION_PROMPT,Instrução:\nVocê é um candidato na prova de um...,383,542,925,D,False,B.,B,D
3,"Pedro, cidadão brasileiro, graduou-se em Direi...",Revalidação do título de graduação em Direito.,Aprovação em Exame de Ordem.,Ter sido admitido em estágio profissional de a...,Prestação de compromisso perante o conselho.,None,4,3,Estatuto da Advocacia e da OAB e Código de Ética,Extração manual arquivos .txt /data/gabaritos-...,...,EXPLAINATION_PROMPT,Instrução:\nVocê é um candidato na prova de um...,270,489,759,A,False,A,A,A
4,O advogado Edson foi contratado para prestar a...,Edson pode prestar a assessoria de modo verbal...,Edson deve prestar a assessoria de modo escrit...,Edson pode prestar a assessoria de modo verbal...,Edson deve prestar a assessoria de modo escrit...,None,5,3,Estatuto da Advocacia e da OAB e Código de Ética,Extração manual arquivos .txt /data/gabaritos-...,...,EXPLAINATION_PROMPT,Instrução:\nVocê é um candidato na prova de um...,317,476,793,C,False,C.,C,B
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
149,"A Escolinha do Gol, entidade privada sem fins ...","Por não se tratar de uma entidade pública, e s...",Não pode o TCU fiscalizar e controlar tais rep...,É devida a prestação de contas de qualquer pes...,Apenas deverão prestar contas dos recursos púb...,None,24,8,Direito Financeiro,Extração manual arquivos .txt /data/gabaritos-...,...,EXPLAINATION_PROMPT,Instrução:\nVocê é um candidato na prova de um...,561,508,1069,C,True,C.,C,C
150,"Marina, empregada doméstica, é casada com Pedr...",Pedro não possui a carência mínima para recebe...,Marina e Pedro não fazem jus ao salário-famíli...,Marina e Pedro têm direito ao benefício do sal...,"Pedro, na qualidade de trabalhador avulso, não...",None,69,19,Direito Previdenciário,Extração manual arquivos .txt /data/gabaritos-...,...,EXPLAINATION_PROMPT,Instrução:\nVocê é um candidato na prova de um...,351,485,836,C,True,C.,C,C
151,Henrique e Amanda foram casados por 30 anos. E...,"Do requerimento, já que foi requerido 60 dias ...","Do óbito, já que foi requerido em até 90 dias ...","Da decisão judicial, já que Henrique perdeu o ...","Do óbito, independentemente da data em que foi...",None,70,20,Direito Previdenciário,Extração manual arquivos .txt /data/gabaritos-...,...,EXPLAINATION_PROMPT,Instrução:\nVocê é um candidato na prova de um...,331,704,1035,B,False,D.,D,B
152,"José, mec

## Resultado Modelo Fine Tuned

In [ ]:
import re
from sklearn.metrics import accuracy_score

def get_answer_from_response(s):
    # gets the first letter of the response
    try:
        return re.search(r'\b[A-E]\b', s).group()
    except:
        return 'ERROR'

test['FINE_TUNED_PHI_RESPONSE_PARSED'] = test['FINE_TUNED_PHI_RESPONSE'].apply(get_answer_from_response)
test['CORRECT'] = test['FINE_TUNED_PHI_RESPONSE_PARSED'] == test['ANSWER']

print('accuracy:',accuracy_score(test['ANSWER'], test['FINE_TUNED_PHI_RESPONSE_PARSED']))

accuracy: 0.525974025974026


In [ ]:
print(test.FINE_TUNED_PHI_RESPONSE_PARSED.value_counts())

FINE_TUNED_PHI_RESPONSE_PARSED
C    39
B    39
A    38
D    38
Name: count, dtype: int64


In [ ]:
test.groupby('NAME_TEST')['CORRECT'].mean()

,CORRECT
NAME_TEST,
39º EXAME DE ORDEM UNIFICADO,0.545455
40º EXAME DE ORDEM UNIFICADO,0.506494


<a name="Save"></a>
### Saving, loading finetuned models
To save the final model as LoRA adapters, either use Huggingface's `push_to_hub` for an online save or `save_pretrained` for a local save.

**[NOTE]** This ONLY saves the LoRA adapters, and not the full model. To save to 16bit or GGUF, scroll down!

In [ ]:
model.save_pretrained("lora_model")
tokenizer.save_pretrained("lora_model")

('lora_model/tokenizer_config.json',
 'lora_model/special_tokens_map.json',
 'lora_model/tokenizer.model',
 'lora_model/added_tokens.json',
 'lora_model/tokenizer.json')